# 20 Build Open Dataset Mix

Builds the open corpus from allowlisted datasets only.

Parameters:
- `SOURCE_MODE`: `"hf"` or `"local"`
- `DRY_RUN`: when true, limits rows per source for fast checks

Required outputs:
- `workspace/reports/open_corpus_build_report.json`
- `workspace/interim/open_sft.jsonl`
- `workspace/interim/open_preferences.jsonl`


In [ ]:

from __future__ import annotations

import json
import uuid
from collections import Counter
from pathlib import Path

import yaml

from lumis1.filters import apply_row_filters
from lumis1.hf_ingest import IngestError, load_allowlist, stream_source_records
from lumis1.license_ledger import validate_allowlist_sources
from lumis1.mixing_math import estimate_row_tokens
from lumis1.schema import validate_preference_row, validate_sft_row

REPO_ROOT = Path.cwd().resolve()
REPORTS = REPO_ROOT / "workspace" / "reports"
INTERIM = REPO_ROOT / "workspace" / "interim"
REPORTS.mkdir(parents=True, exist_ok=True)
INTERIM.mkdir(parents=True, exist_ok=True)

SOURCE_MODE = "hf"  # 'hf' or 'local'
DRY_RUN = False
STREAMING = True

allowlist_path = REPO_ROOT / "configs" / "dataset_sources_allowlist.yaml"
mixture_path = REPO_ROOT / "configs" / "dataset_mixture.yaml"
identity_report_path = REPORTS / "identity_validation.json"

allowlist_raw = yaml.safe_load(allowlist_path.read_text(encoding="utf-8")) or {}
mixture = yaml.safe_load(mixture_path.read_text(encoding="utf-8")) or {}
sources = [s for s in allowlist_raw.get("sources", []) if isinstance(s, dict)]
validate_allowlist_sources(sources)
allowlist_map = load_allowlist(allowlist_path)

mixture_sources = [s for s in mixture.get("sources", []) if isinstance(s, dict)]
mixture_source_map = {str(s.get("source_id")): s for s in mixture_sources if s.get("source_id")}

if not identity_report_path.exists():
    raise FileNotFoundError(
        "Missing workspace/reports/identity_validation.json. Run notebook 10 first."
    )
identity_report = json.loads(identity_report_path.read_text(encoding="utf-8"))
identity_tokens = int(identity_report.get("tokens", {}).get("sft_tokens_total", 0))
if identity_tokens <= 0:
    raise RuntimeError("identity token count missing or zero in identity_validation.json")

identity_token_share = float(mixture.get("identity_pack", {}).get("fixed_share_of_final_sft_tokens", 0.20))
if not 0 < identity_token_share < 1:
    raise RuntimeError(f"invalid identity token share target: {identity_token_share}")

open_token_budget_target = int(identity_tokens * ((1.0 - identity_token_share) / identity_token_share))
dry_run_budget = int(mixture.get("token_budgets", {}).get("open_sft_budget", {}).get("dry_run_tokens", 50000))
if DRY_RUN:
    open_token_budget_target = min(open_token_budget_target, dry_run_budget)

scan_limit_default = int(mixture.get("ingestion_defaults", {}).get("max_records_scanned_per_source", 2000000))
if DRY_RUN:
    scan_limit_default = min(scan_limit_default, 20000)

def canonical_messages(rec: dict) -> list[dict]:
    if isinstance(rec.get("messages"), list) and rec["messages"]:
        return rec["messages"]
    prompt = rec.get("prompt") or rec.get("instruction") or rec.get("question") or rec.get("input")
    answer = rec.get("response") or rec.get("output") or rec.get("answer") or rec.get("chosen")
    if not isinstance(prompt, str) or not prompt.strip():
        prompt = "No prompt provided"
    if not isinstance(answer, str) or not answer.strip():
        answer = "No response provided"
    return [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": answer},
    ]

def extract_preference_triplet(rec: dict) -> tuple[str, str, str] | None:
    prompt = rec.get("prompt") or rec.get("instruction") or rec.get("question") or rec.get("input")
    chosen = rec.get("chosen")
    rejected = rec.get("rejected")
    if isinstance(prompt, str) and isinstance(chosen, str) and isinstance(rejected, str):
        return prompt, chosen, rejected

    # Pairwise formats with winner indicator.
    a = rec.get("response_a") or rec.get("answer_a") or rec.get("candidate_a")
    b = rec.get("response_b") or rec.get("answer_b") or rec.get("candidate_b")
    label = rec.get("label") or rec.get("preference") or rec.get("winner")
    if isinstance(prompt, str) and isinstance(a, str) and isinstance(b, str) and isinstance(label, str):
        upper = label.strip().upper()
        if upper in {"A", "LEFT", "1", "CHOICE_A"}:
            return prompt, a, b
        if upper in {"B", "RIGHT", "2", "CHOICE_B"}:
            return prompt, b, a

    return None

open_sft: list[dict] = []
open_prefs: list[dict] = []
source_reports: list[dict] = []

global_open_tokens = 0
global_remaining = open_token_budget_target

for source_id, source_cfg in mixture_source_map.items():
    if global_remaining <= 0:
        break

    allow_entry = allowlist_map.get(source_id)
    if not isinstance(allow_entry, dict):
        source_reports.append({"source_id": source_id, "error": "source missing in allowlist"})
        continue
    if allow_entry.get("enabled") is not True:
        source_reports.append({"source_id": source_id, "skipped": "disabled_in_allowlist"})
        continue
    if allow_entry.get("source_mode") != SOURCE_MODE:
        source_reports.append({"source_id": source_id, "skipped": "source_mode_mismatch"})
        continue

    source_budget = int(source_cfg.get("token_budget") or allow_entry.get("max_tokens_cap") or 0)
    if source_budget <= 0:
        source_reports.append({"source_id": source_id, "skipped": "token_budget_zero"})
        continue

    source_remaining = min(source_budget, global_remaining)
    source_kept_tokens = 0
    source_kept_rows = 0
    raw_rows_scanned = 0
    pref_kept_rows = 0
    drop_reasons = Counter()

    source_entry = dict(allow_entry)
    source_entry["split"] = str(allow_entry.get("default_split") or "train")

    try:
        stream = stream_source_records(
            source_entry,
            source_mode=SOURCE_MODE,
            allowlist=allowlist_map,
            limit=scan_limit_default,
            streaming=STREAMING,
        )
    except IngestError as exc:
        source_reports.append({"source_id": source_id, "error": str(exc)})
        continue

    for rec in stream:
        if source_remaining <= 0 or global_remaining <= 0:
            break

        raw_rows_scanned += 1
        base = {
            "schema_version": "1.0",
            "id": str(rec.get("id") or uuid.uuid4()),
            "source_id": source_id,
            "license": str(allow_entry.get("license", "")),
            "thinking": "off",
            "chat_template_kwargs": {"enable_thinking": False},
            "category": str(source_cfg.get("category") or "utility_tasks"),
            "modality": str(source_cfg.get("modality") or "text"),
            "messages": canonical_messages(rec),
        }

        filtered_rows, filt_report = apply_row_filters([base], drop_on_cot=True)
        for reason, count in (filt_report.get("drop_reasons") or {}).items():
            drop_reasons[reason] += int(count)
        if not filtered_rows:
            continue

        try:
            row = validate_sft_row(filtered_rows[0])
        except Exception as exc:  # noqa: BLE001
            drop_reasons[f"schema:{str(exc).splitlines()[0][:120]}"] += 1
            continue

        row_tokens = estimate_row_tokens(row)
        if row_tokens <= 0:
            drop_reasons["zero_token_row"] += 1
            continue
        if row_tokens > source_remaining:
            drop_reasons["source_token_budget_skip"] += 1
            continue
        if row_tokens > global_remaining:
            drop_reasons["global_token_budget_skip"] += 1
            continue

        open_sft.append(row)
        source_kept_tokens += row_tokens
        source_kept_rows += 1
        source_remaining -= row_tokens
        global_remaining -= row_tokens
        global_open_tokens += row_tokens

        pref_triplet = extract_preference_triplet(rec)
        if pref_triplet and source_id in set(mixture.get("preferences", {}).get("enabled_sources", [])):
            prompt, chosen, rejected = pref_triplet
            pref_row = {
                "id": str(rec.get("id") or uuid.uuid4()),
                "source_id": source_id,
                "license": str(allow_entry.get("license", "")),
                "thinking": "off",
                "chat_template_kwargs": {"enable_thinking": False},
                "prompt": prompt,
                "chosen": chosen,
                "rejected": rejected,
            }
            try:
                open_prefs.append(validate_preference_row(pref_row))
                pref_kept_rows += 1
            except Exception:  # noqa: BLE001
                drop_reasons["pref_schema_invalid"] += 1

    source_reports.append(
        {
            "source_id": source_id,
            "token_budget": source_budget,
            "kept_tokens": source_kept_tokens,
            "raw_rows_scanned": raw_rows_scanned,
            "kept_rows": source_kept_rows,
            "kept_preference_rows": pref_kept_rows,
            "drop_reasons": dict(drop_reasons),
            "source_budget_fully_used": source_remaining == 0,
        }
    )

if global_open_tokens != open_token_budget_target:
    raise RuntimeError(
        f"open token budget mismatch: expected {open_token_budget_target}, got {global_open_tokens}"
    )

sft_path = INTERIM / "open_sft.jsonl"
pref_path = INTERIM / "open_preferences.jsonl"
with sft_path.open("w", encoding="utf-8") as f:
    for row in open_sft:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
with pref_path.open("w", encoding="utf-8") as f:
    for row in open_prefs:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

report = {
    "source_mode": SOURCE_MODE,
    "dry_run": DRY_RUN,
    "streaming": STREAMING,
    "open_sft_token_budget_target": open_token_budget_target,
    "open_sft_tokens_built": global_open_tokens,
    "sources": source_reports,
    "totals": {
        "open_sft_rows": len(open_sft),
        "open_preference_rows": len(open_prefs),
    },
    "outputs": {
        "open_sft": str(sft_path),
        "open_preferences": str(pref_path),
    },
}

report_path = REPORTS / "open_corpus_build_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))
print(f"saved: {report_path}")
